In [1]:
# cell 1 — imports and connect
import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
from neo4j import GraphDatabase
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

NEO4J_URI  = "neo4j+s://b8f5be60.databases.neo4j.io"
NEO4J_USER = "b8f5be60"
NEO4J_PASS = "lRDCVmyEOwRRbmproEd-7y30MilsPPh2uGbHSAIQo_U"
NEO4J_DB   = "b8f5be60"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with driver.session(database=NEO4J_DB) as s:
    msg = s.run("RETURN 'connected' AS msg").single()['msg']
print(f'neo4j  : {msg}')

mongo    = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017'))
docs_col = mongo['csai415_rag']['documents']
print(f'mongodb: {docs_col.count_documents({})} documents ready')

neo4j  : connected
mongodb: 144 documents ready


In [2]:
# cell 2 — helper and schema setup
NEO4J_DB = "b8f5be60"

def run(q, **p):
    with driver.session(database=NEO4J_DB) as s:
        return [dict(r) for r in s.run(q, **p)]

run('CREATE CONSTRAINT paper_doc_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.doc_id IS UNIQUE')
run('CREATE CONSTRAINT author_name_norm IF NOT EXISTS FOR (a:Author) REQUIRE a.name_norm IS UNIQUE')
run('CREATE CONSTRAINT topic_name IF NOT EXISTS FOR (t:Topic) REQUIRE t.name IS UNIQUE')
run('CREATE INDEX paper_year IF NOT EXISTS FOR (p:Paper) ON (p.year)')
run('CREATE INDEX paper_venue IF NOT EXISTS FOR (p:Paper) ON (p.venue)')

print('schema ready')

schema ready


In [3]:
# cell 3 — load Paper nodes from MongoDB into Neo4j
papers = list(docs_col.find({}, {'_id': 0}))
print(f'loading {len(papers)} papers into neo4j...')

for p in tqdm(papers, desc='Paper nodes'):
    run("""
        MERGE (n:Paper {doc_id: $doc_id})
        SET n.title    = $title,
            n.year     = $year,
            n.venue    = $venue,
            n.abstract = $abstract,
            n.status   = 'ingested',
            n.authors  = $authors
    """,
    doc_id   = p.get('doc_id', ''),
    title    = p.get('title', ''),
    year     = int(p.get('year', 0)),
    venue    = p.get('venue', ''),
    abstract = str(p.get('abstract', ''))[:500],
    authors  = p.get('authors', [])
    )

count = run('MATCH (p:Paper) RETURN count(p) AS n')[0]['n']
print(f'Paper nodes in Neo4j: {count}')

loading 144 papers into neo4j...


Paper nodes:   0%|          | 0/144 [00:00<?, ?it/s]

Paper nodes in Neo4j: 144


In [4]:
# cell 4 — create Author nodes and WROTE edges
def norm_name(name):
    """normalise author name for deduplication — handles Last, First format"""
    if not name or not name.strip():
        return ''
    if ',' in name:
        parts = name.split(',', 1)
        name  = f'{parts[1].strip()} {parts[0].strip()}'
    name = name.lower()
    name = re.sub(r'[^a-z\s]', '', name)
    return re.sub(r'\s+', ' ', name).strip()

for p in tqdm(papers, desc='Author nodes + WROTE edges'):
    for order, author in enumerate(p.get('authors', []), start=1):
        nn = norm_name(author)
        if not nn:
            continue
        run("""
            MERGE (a:Author {name_norm: $nn})
            SET a.name = $name
            WITH a
            MATCH (paper:Paper {doc_id: $doc_id})
            MERGE (a)-[:WROTE {order: $order}]->(paper)
        """, nn=nn, name=author, doc_id=p['doc_id'], order=order)

a_count = run('MATCH (a:Author) RETURN count(a) AS n')[0]['n']
w_count = run('MATCH ()-[:WROTE]->() RETURN count(*) AS n')[0]['n']
print(f'Author nodes : {a_count}')
print(f'WROTE edges  : {w_count}')

Author nodes + WROTE edges:   0%|          | 0/144 [00:00<?, ?it/s]

Author nodes : 50
WROTE edges  : 50


In [5]:
# cell 5 — create Topic nodes and ABOUT edges (from arxiv categories)
for p in tqdm(papers, desc='Topic nodes + ABOUT edges'):
    for kw in p.get('keywords', [])[:3]:   # top 3 arxiv categories per paper
        topic = kw.strip()
        if not topic:
            continue
        run("""
            MERGE (t:Topic {name: $topic})
            WITH t
            MATCH (paper:Paper {doc_id: $doc_id})
            MERGE (paper)-[:ABOUT]->(t)
        """, topic=topic, doc_id=p['doc_id'])

t_count = run('MATCH (t:Topic) RETURN count(t) AS n')[0]['n']
ab_count = run('MATCH ()-[:ABOUT]->() RETURN count(*) AS n')[0]['n']
print(f'Topic nodes  : {t_count}')
print(f'ABOUT edges  : {ab_count}')

Topic nodes + ABOUT edges:   0%|          | 0/144 [00:00<?, ?it/s]

Topic nodes  : 5
ABOUT edges  : 144


In [6]:
# cell 6 — full graph summary
with driver.session() as s:
    paper_n = s.run('MATCH (p:Paper) RETURN count(p) AS n').single()['n']
    author_n = s.run('MATCH (a:Author) RETURN count(a) AS n').single()['n']
    topic_n = s.run('MATCH (t:Topic) RETURN count(t) AS n').single()['n']
    wrote_n = s.run('MATCH ()-[:WROTE]->() RETURN count(*) AS n').single()['n']
    about_n = s.run('MATCH ()-[:ABOUT]->() RETURN count(*) AS n').single()['n']

print('\n=== NEO4J GRAPH SUMMARY ===')
print(f'  Paper nodes   : {paper_n}')
print(f'  Author nodes  : {author_n}')
print(f'  Topic nodes   : {topic_n}')
print(f'  WROTE edges   : {wrote_n}')
print(f'  ABOUT edges   : {about_n}')


=== NEO4J GRAPH SUMMARY ===
  Paper nodes   : 144
  Author nodes  : 50
  Topic nodes   : 5
  WROTE edges   : 50
  ABOUT edges   : 144


In [7]:
# cell 7 — CYPHER QUERY 1: most prolific authors in corpus
print('=' * 60)
print('CYPHER QUERY 1: Most prolific authors')
print('=' * 60)
rows = run("""
    MATCH (a:Author)-[:WROTE]->(p:Paper)
    RETURN a.name AS author, count(p) AS papers
    ORDER BY papers DESC
    LIMIT 10
""")
for r in rows:
    print(f"  {r['papers']:>3}x  {r['author']}")

CYPHER QUERY 1: Most prolific authors
    1x  Md. Mahdi Mohtasim
    1x  Shakil Mosharrof
    1x  T. Gopi Krishna
    1x  Aniruddha Salve
    1x  Saba Attar
    1x  Mahesh Deshmukh
    1x  Sayali Shivpuje
    1x  Arnab Mitra Utsab
    1x  Ali Naseh
    1x  Nurshat Fateh Ali


In [8]:
# cell 8 — CYPHER QUERY 2: papers per year
print('=' * 60)
print('CYPHER QUERY 2: Papers per year (2021 onwards)')
print('=' * 60)
rows = run("""
    MATCH (p:Paper)
    WHERE p.year >= 2021
    RETURN p.year AS year, count(p) AS papers
    ORDER BY year
""")
for r in rows:
    bar = '█' * r['papers']
    print(f"  {r['year']}: {r['papers']:>3} papers  {bar}")

CYPHER QUERY 2: Papers per year (2021 onwards)
  2021:  26 papers  ██████████████████████████
  2022:  34 papers  ██████████████████████████████████
  2023:  43 papers  ███████████████████████████████████████████
  2024:  21 papers  █████████████████████
  2025:   5 papers  █████
  2026:   3 papers  ███


In [9]:
# cell 9 — CYPHER QUERY 3: most common research topics
print('=' * 60)
print('CYPHER QUERY 3: Most common research topics (arxiv categories)')
print('=' * 60)
rows = run("""
    MATCH (p:Paper)-[:ABOUT]->(t:Topic)
    RETURN t.name AS topic, count(p) AS papers
    ORDER BY papers DESC
    LIMIT 10
""")
for r in rows:
    print(f"  {r['papers']:>3}x  {r['topic']}")

CYPHER QUERY 3: Most common research topics (arxiv categories)
  138x  cs.IR
    3x  cs.CL
    1x  cs.AI
    1x  cs.CR
    1x  cs.LG


In [10]:
# cell 10 — CYPHER QUERY 4: authors who published across multiple topics
print('=' * 60)
print('CYPHER QUERY 4: Authors spanning multiple topics')
print('=' * 60)
rows = run("""
    MATCH (a:Author)-[:WROTE]->(p:Paper)-[:ABOUT]->(t:Topic)
    WITH a, count(DISTINCT t) AS topic_count
    WHERE topic_count > 1
    RETURN a.name AS author, topic_count
    ORDER BY topic_count DESC
    LIMIT 10
""")
for r in rows:
    print(f"  {r['author']}: {r['topic_count']} topics")

CYPHER QUERY 4: Authors spanning multiple topics


In [11]:
# cell 11 — CYPHER QUERY 5: co-authorship pairs
print('=' * 60)
print('CYPHER QUERY 5: Co-authorship pairs (shared papers)')
print('=' * 60)
rows = run("""
    MATCH (a1:Author)-[:WROTE]->(p:Paper)<-[:WROTE]-(a2:Author)
    WHERE a1.name_norm < a2.name_norm
    RETURN a1.name AS author1, a2.name AS author2, count(p) AS shared_papers
    ORDER BY shared_papers DESC
    LIMIT 10
""")
if rows:
    for r in rows:
        print(f"  {r['shared_papers']}x  {r['author1']}  <->  {r['author2']}")
else:
    print('  (no co-authorship pairs found — authors appear on different papers)')

CYPHER QUERY 5: Co-authorship pairs (shared papers)
  1x  Ali Naseh  <->  Amir Houmansadr
  1x  Alina Oprea  <->  Amir Houmansadr
  1x  Ali Naseh  <->  Anshuman Suri
  1x  Alina Oprea  <->  Anshuman Suri
  1x  Amir Houmansadr  <->  Anshuman Suri
  1x  Aniruddha Salve  <->  Arnab Mitra Utsab
  1x  Arbi Haza Nasution  <->  Aytug Onan
  1x  Alireza Seddighi  <->  Dave Nielson
  1x  Dave Nielson  <->  Dean Wampler
  1x  Ali Naseh  <->  Alina Oprea


In [12]:
# cell 12 — close connection
driver.close()
mongo.close()
print('done — neo4j and mongodb connections closed')

done — neo4j and mongodb connections closed
